# Tarea 1 Cadenas de Markov - Simulación de Monte Carlo

## Integrantes

- Nicolás Betancur Sánchez 1010023513
- Andrés Felipe Gómez Durán 1001298960
- Germán Camilo Rodríguez Perilla 1032480170

## Planteamiento general

Se considera el espacio muestral

$$
\Omega=[-2,2]\times[-2,2],
$$

cuya área es $|\Omega|=16$. Si un punto $(X,Y)$ se distribuye uniformemente en $\Omega$, la probabilidad de pertenecer a un conjunto medible $A\subseteq\Omega$ es

$$
P\big((X,Y)\in A\big)=\frac{|A|}{|\Omega|}.
$$

## Método de simulación

Se emplea el método de Monte Carlo: se generan puntos pseudoaleatorios uniformes en $\Omega$, se registra cuántos satisfacen la condición de pertenencia al círculo y se usa la frecuencia relativa como estimación de la probabilidad. Para que los resultados puedan reproducirse, se fija la semilla $20260909$.

## 1. Aproximar la probabilidad $P(C_1)$

Sea

$$
C_1=\left\{(x,y)\in\Omega:x^2+y^2\leq 1\right\}.
$$

### 1.1 Generación de puntos uniformes

Para cada tamaño de muestra $n$, se generan pares independientes

$$
X_j\sim \operatorname{Unif}(-2,2),
\qquad
Y_j\sim \operatorname{Unif}(-2,2),
\qquad j=1,\ldots,n.
$$

La independencia de las coordenadas hace que $(X_j,Y_j)$ sea uniforme en todo el cuadrado $\Omega$.

### 1.2 Identificación de los puntos dentro del círculo

Para cada punto se calcula la variable indicadora

$$
I_j=\mathbf{1}\left\{X_j^2+Y_j^2\leq 1\right\}.
$$

Así, $I_j=1$ si el punto está dentro de $C_1$ y $I_j=0$ en caso contrario. El número de puntos dentro del círculo es $N_1=\sum_{j=1}^n I_j$.

### 1.3 Estimación de la probabilidad y del error

La frecuencia relativa

$$
\widehat{P}_n(C_1)=\frac{N_1}{n}
=\frac{1}{n}\sum_{j=1}^n I_j
$$

es un estimador insesgado de $P(C_1)$. Como el círculo unitario tiene área $\pi$ y el cuadrado tiene área $16$,

$$
P(C_1)=\frac{\pi}{16}\approx 0.19634954.
$$

El error absoluto se calcula mediante

$$
\left|P(C_1)-\widehat{P}_n(C_1)\right|.
$$

Por la ley de los grandes números, $\widehat{P}_n(C_1)\to P(C_1)$ cuando $n\to\infty$.

### 1.4 Tamaños de muestra

La simulación se realiza para

$$
n\in\left\{10^2,10^3,10^4,10^5,10^6\right\}.
$$

Para cada valor se reportan $n$, el número de puntos dentro del círculo, $\widehat{P}_n(C_1)$ y el error absoluto.

### 1.5 Implementación y resultados

In [ ]:
import numpy as np
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj)

SEMILLA = 20260909
TAMANOS = [10**i for i in range(2, 7)]
P_C1 = np.pi / 16

def generar_distancias_cuadradas(n, semilla):
    """Genera n puntos uniformes en [-2, 2]^2 y retorna x^2 + y^2."""
    rng = np.random.default_rng(semilla)
    puntos = rng.uniform(-2, 2, size=(n, 2))
    return np.einsum("ij,ij->i", puntos, puntos)

# Para cada n se usa una muestra reproducible. Estas mismas muestras se
# reutilizan en el ejercicio 3 para comparar justamente los tres radios.
muestras_d2 = {
    n: generar_distancias_cuadradas(n, SEMILLA + int(np.log10(n)))
    for n in TAMANOS
}

registros_ej1 = []
for n, distancias2 in muestras_d2.items():
    puntos_dentro = int(np.count_nonzero(distancias2 <= 1))
    probabilidad_estimada = puntos_dentro / n
    registros_ej1.append(
        {
            "n": n,
            "Puntos dentro de C1": puntos_dentro,
            "Probabilidad estimada": probabilidad_estimada,
            "Error absoluto": abs(P_C1 - probabilidad_estimada),
        }
    )

tabla_ej1 = pd.DataFrame(registros_ej1)
display(
    tabla_ej1.round(
        {"Probabilidad estimada": 8, "Error absoluto": 8}
    )
)

### 1.6 Tabla de resultados

| $n$ | Puntos dentro de $C_1$ | $\widehat{P}_n(C_1)$ | Error absoluto |
|---:|---:|---:|---:|
| $10^2$ | 19 | 0.19000000 | 0.00634954 |
| $10^3$ | 202 | 0.20200000 | 0.00565046 |
| $10^4$ | 1 950 | 0.19500000 | 0.00134954 |
| $10^5$ | 19 647 | 0.19647000 | 0.00012046 |
| $10^6$ | 196 575 | 0.19657500 | 0.00022546 |

### 1.7 Análisis de los resultados

Las estimaciones se concentran alrededor del valor teórico $\pi/16\approx 0.19634954$ a medida que aumenta $n$. La reducción del error no tiene que ser monótona: cada estimación contiene variación aleatoria y, por casualidad, una muestra menor puede quedar más cerca del valor real que una muestra mayor. Lo relevante es que la dispersión típica disminuye a la tasa $1/\sqrt{n}$, puesto que

$$
\operatorname{Var}\!\left(\widehat{P}_n(C_1)\right)
=\frac{P(C_1)\left[1-P(C_1)\right]}{n}.
$$

Con $n=10^6$, el error observado es aproximadamente $2.25\times 10^{-4}$, lo cual evidencia la estabilización de la frecuencia relativa.

## 2. Aproximar el número $\pi$

### 2.1 Relación entre $P(C_1)$ y $\pi$

La relación geométrica obtenida en el ejercicio anterior es

$$
P(C_1)=\frac{\text{área del círculo unitario}}
{\text{área del cuadrado}}
=\frac{\pi}{16}.
$$

Al despejar $\pi$ se obtiene

$$
\pi=16P(C_1).
$$

### 2.2 Construcción y justificación del estimador

Al reemplazar la probabilidad desconocida por su frecuencia relativa se define

$$
\widehat{\pi}_n=16\widehat{P}_n(C_1).
$$

Este estimador es insesgado, porque

$$
E\!\left[\widehat{\pi}_n\right]
=16E\!\left[\widehat{P}_n(C_1)\right]
=16P(C_1)=\pi.
$$

También es consistente: como $\widehat{P}_n(C_1)\to P(C_1)$ por la ley de los grandes números, entonces $\widehat{\pi}_n\to\pi$.

### 2.3 Cálculo del error

Para cada tamaño de muestra se calcula

$$
\left|\pi-\widehat{\pi}_n\right|
=
\left|\pi-16\widehat{P}_n(C_1)\right|.
$$

Se reutilizan exactamente las probabilidades estimadas en el ejercicio 1, tal como solicita el enunciado.

### 2.4 Implementación y resultados

In [ ]:
tabla_ej2 = tabla_ej1[["n", "Probabilidad estimada"]].copy()
tabla_ej2["Pi estimado"] = 16 * tabla_ej2["Probabilidad estimada"]
tabla_ej2["Error absoluto de Pi"] = abs(np.pi - tabla_ej2["Pi estimado"])

display(
    tabla_ej2.round(
        {"Probabilidad estimada": 8, "Pi estimado": 8, "Error absoluto de Pi": 8}
    )
)

### 2.5 Tabla de resultados

| $n$ | $\widehat{P}_n(C_1)$ | $\widehat{\pi}_n$ | Error absoluto |
|---:|---:|---:|---:|
| $10^2$ | 0.19000000 | 3.04000000 | 0.10159265 |
| $10^3$ | 0.20200000 | 3.23200000 | 0.09040735 |
| $10^4$ | 0.19500000 | 3.12000000 | 0.02159265 |
| $10^5$ | 0.19647000 | 3.14352000 | 0.00192735 |
| $10^6$ | 0.19657500 | 3.14520000 | 0.00360735 |

### 2.6 Análisis de los resultados

El factor $16$ transforma directamente el error de la probabilidad:

$$
\left|\pi-\widehat{\pi}_n\right|
=16\left|P(C_1)-\widehat{P}_n(C_1)\right|.
$$

Por eso las aproximaciones de $\pi$ siguen el mismo patrón de convergencia observado en el ejercicio 1. En esta realización, $n=10^5$ produce una estimación más cercana que $n=10^6$; esto no contradice la convergencia, pues esta es una propiedad de largo plazo y no exige que el error disminuya en cada paso. Para muestras grandes, las estimaciones se mantienen alrededor de $3.14$.

## 3. Considerar los conjuntos $C_{\sqrt{2}}$ y $C_2$

Se definen

$$
C_{\sqrt{2}}
=\left\{(x,y)\in\Omega:x^2+y^2\leq 2\right\},
\qquad
C_2
=\left\{(x,y)\in\Omega:x^2+y^2\leq 2^2\right\}.
$$

### 3.1 Probabilidades teóricas

Para un círculo de radio $r\leq 2$ contenido en $\Omega$, la probabilidad de pertenencia es

$$
P(C_r)=\frac{\pi r^2}{16}.
$$

En particular,

$$
P(C_{\sqrt{2}})=\frac{\pi(\sqrt{2})^2}{16}
=\frac{\pi}{8}
\approx 0.39269908,
$$

y

$$
P(C_2)=\frac{\pi(2)^2}{16}
=\frac{\pi}{4}
\approx 0.78539816.
$$

### 3.2 Identificación de los puntos

Para cada radio se utiliza la variable indicadora

$$
I_j^{(r)}
=\mathbf{1}\left\{X_j^2+Y_j^2\leq r^2\right\}.
$$

La estimación de la probabilidad es

$$
\widehat{P}_n(C_r)
=\frac{1}{n}\sum_{j=1}^n I_j^{(r)}.
$$

### 3.3 Estimador general de $\pi$

De $P(C_r)=\pi r^2/16$ se despeja $\pi$ y se obtiene

$$
\widehat{\pi}_n(r)
=\frac{16}{r^2}\widehat{P}_n(C_r).
$$

El estimador es insesgado y consistente. Además,

$$
\operatorname{Var}\!\left(\widehat{\pi}_n(r)\right)
=\frac{256}{r^4}
\frac{P(C_r)\left[1-P(C_r)\right]}{n}
=\frac{16\pi/r^2-\pi^2}{n}.
$$

Esta varianza disminuye al aumentar $r$ dentro del intervalo considerado, lo cual anticipa que el radio $r=2$ debe proporcionar, en promedio, estimaciones más precisas.

### 3.4 Comparación de los tres radios

Para cada $n$ se reutiliza la misma nube de puntos en los radios $1$, $\sqrt{2}$ y $2$. De esta manera, la diferencia entre estimadores se debe al criterio de pertenencia y no a haber usado muestras distintas.

### 3.5 Implementación y resultados

In [ ]:
configuracion_radios = [
    ("1", 1.0),
    ("sqrt(2)", np.sqrt(2)),
    ("2", 2.0),
]

registros_ej3 = []
for n, distancias2 in muestras_d2.items():
    for etiqueta, radio in configuracion_radios:
        puntos_dentro = int(np.count_nonzero(distancias2 <= radio**2))
        probabilidad_estimada = puntos_dentro / n
        probabilidad_teorica = np.pi * radio**2 / 16
        pi_estimado = (16 / radio**2) * probabilidad_estimada

        registros_ej3.append(
            {
                "n": n,
                "Radio": etiqueta,
                "Puntos dentro": puntos_dentro,
                "Probabilidad teórica": probabilidad_teorica,
                "Probabilidad estimada": probabilidad_estimada,
                "Pi estimado": pi_estimado,
                "Error absoluto de Pi": abs(np.pi - pi_estimado),
            }
        )

tabla_ej3_detalle = pd.DataFrame(registros_ej3)

tabla_comparativa = (
    tabla_ej3_detalle
    .pivot(index="n", columns="Radio", values="Pi estimado")
    .reset_index()[["n", "1", "sqrt(2)", "2"]]
    .rename(
        columns={
            "1": "Pi estimado (r=1)",
            "sqrt(2)": "Pi estimado (r=sqrt(2))",
            "2": "Pi estimado (r=2)",
        }
    )
)

columnas_decimales = {
    "Probabilidad teórica": 8,
    "Probabilidad estimada": 8,
    "Pi estimado": 8,
    "Error absoluto de Pi": 8,
}
display(
    tabla_ej3_detalle.loc[tabla_ej3_detalle["Radio"] != "1"]
    .round(columnas_decimales)
)
display(
    tabla_comparativa.round(
        {
            "Pi estimado (r=1)": 8,
            "Pi estimado (r=sqrt(2))": 8,
            "Pi estimado (r=2)": 8,
        }
    )
)

### 3.6 Resultados para $r=\sqrt{2}$ y $r=2$

| Radio | $n$ | Puntos dentro | $\widehat{P}_n(C_r)$ | $\widehat{\pi}_n(r)$ | Error absoluto |
|:---:|---:|---:|---:|---:|---:|
| $\sqrt{2}$ | $10^2$ | 33 | 0.33000000 | 2.64000000 | 0.50159265 |
| $\sqrt{2}$ | $10^3$ | 391 | 0.39100000 | 3.12800000 | 0.01359265 |
| $\sqrt{2}$ | $10^4$ | 3 911 | 0.39110000 | 3.12880000 | 0.01279265 |
| $\sqrt{2}$ | $10^5$ | 39 321 | 0.39321000 | 3.14568000 | 0.00408735 |
| $\sqrt{2}$ | $10^6$ | 392 305 | 0.39230500 | 3.13844000 | 0.00315265 |
| $2$ | $10^2$ | 75 | 0.75000000 | 3.00000000 | 0.14159265 |
| $2$ | $10^3$ | 781 | 0.78100000 | 3.12400000 | 0.01759265 |
| $2$ | $10^4$ | 7 852 | 0.78520000 | 3.14080000 | 0.00079265 |
| $2$ | $10^5$ | 78 550 | 0.78550000 | 3.14200000 | 0.00040735 |
| $2$ | $10^6$ | 785 025 | 0.78502500 | 3.14010000 | 0.00149265 |

### 3.7 Tabla comparativa de las aproximaciones de $\pi$

| $n$ | $\widehat{\pi}_n(r=1)$ | $\widehat{\pi}_n(r=\sqrt{2})$ | $\widehat{\pi}_n(r=2)$ |
|---:|---:|---:|---:|
| $10^2$ | 3.04000000 | 2.64000000 | 3.00000000 |
| $10^3$ | 3.23200000 | 3.12800000 | 3.12400000 |
| $10^4$ | 3.12000000 | 3.12880000 | 3.14080000 |
| $10^5$ | 3.14352000 | 3.14568000 | 3.14200000 |
| $10^6$ | 3.14520000 | 3.13844000 | 3.14010000 |

### 3.8 Comparación y análisis

Los tres estimadores convergen a $\pi$, pero no tienen la misma variabilidad. La expresión

$$
\operatorname{Var}\!\left(\widehat{\pi}_n(r)\right)
=\frac{16\pi/r^2-\pi^2}{n}
$$

muestra que, para un mismo $n$, la varianza es menor cuando el radio es mayor. En particular, las desviaciones estándar teóricas son aproximadamente

$$
\frac{6.36}{\sqrt{n}}\quad(r=1),\qquad
\frac{3.91}{\sqrt{n}}\quad(r=\sqrt{2}),\qquad
\frac{1.64}{\sqrt{n}}\quad(r=2).
$$

Por tanto, $r=2$ es teóricamente la opción más eficiente entre los tres radios. Los resultados numéricos son coherentes con esta conclusión: para $n=10^4$, $10^5$ y $10^6$, el estimador basado en $r=2$ presenta errores absolutos de $0.00079265$, $0.00040735$ y $0.00149265$, respectivamente, generalmente menores que los obtenidos con los otros radios.

Aunque en una realización particular otro radio puede producir accidentalmente un error menor, el radio $2$ utiliza una mayor proporción del cuadrado y reduce la amplificación del error al aplicar el factor $16/r^2$. Esta es la razón estadística por la que ofrece mejores aproximaciones en promedio.